<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Chapter 12 · Short-Rate Models: Vasicek, CIR, and Hull-White

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/ch12_short_rate_models_vasicek_cir_hull_white.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

Short-rate models compress the whole curve into one state variable and specify how it evolves. In this notebook you reproduce the chapter's Euler simulations of Vasicek, CIR, and Hull-White-style paths from frozen parameters, compare one-year teaching discount factors estimated by Monte Carlo, and read the grid-and-seed diagnostics. You learn to state the exact conditional mean and variance of the Vasicek rate, to explain why the normal models can produce negative rates while the CIR steps are projected to stay non-negative, and to keep model scenarios strictly separate from market forecasts.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Run the chapter example

The chapter script simulates short-rate paths for Vasicek, CIR, and a Hull-White-style teaching process from the frozen parameter file, then estimates a one-year zero-coupon discount factor by Monte Carlo for each model. It prints the mean final rate, the minimum rate, the discount-factor estimate with its standard error, and a grid-and-seed sensitivity table for the Vasicek case on monthly and weekly grids under three seeds.

In [ ]:
SCRIPT = ROOT / "code" / "ch12_short_rate_models.py"
print(SCRIPT.relative_to(ROOT))

result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

## Interpretation checklist

Check the diagnostics against the model economics before quoting any number.

- Confirm the units: kappa, theta, and sigma are annualized, rates are decimals, and the zero-coupon output is a discount factor per unit notional.
- Read the minimum-rate column: in this seeded run all minima are positive, but the normal models are not constrained to stay there, while the CIR steps are explicitly projected to non-negative values.
- Interpret the standard errors: they measure the sampling noise of the Monte Carlo estimate, not time-discretization or model error.
- Check the grid diagnostics: monthly and weekly grids under seeds 101, 202, and 303 all give estimates near 0.9644 to 0.9646, so the answer is stable within sampling noise.
- Remember that no risk-neutral calibration is established: the printed zero prices are teaching discount factors, and the Hull-White-style target is an interpolated zero-rate proxy, not a curve fit.

In [ ]:
import pandas as pd

DATA = ROOT / "data"
chapter_files = sorted(DATA.glob("ch12_*.csv"))
summary = pd.DataFrame([
    {"file": path.name,
     "rows": len(frame := pd.read_csv(path)),
     "columns": len(frame.columns),
     "missing_cells": int(frame.isna().sum().sum()),
     "fields": str(list(frame.columns))}
    for path in chapter_files
])
summary


## Chapter-specific checks

Compare the model assumptions and positivity behavior. A calibrated Hull-White model can fit an initial curve, but the companion Hull-White-style path uses only a zero-rate-target proxy and does not fit that curve.

In [ ]:
params = pd.read_csv(DATA / "ch12_short_rate_params.csv")
params["state_variable"] = "short rate"
params["positivity"] = params.model.map({
    "vasicek": "not guaranteed",
    "cir": "parameter-dependent",
    "hull-white": "not guaranteed",
})
params["main_use"] = params.model.map({
    "vasicek": "transparent baseline",
    "cir": "positive-rate scenarios",
    "hull-white": "zero-rate target proxy (not curve-fit)",
})
params

## Mini-lab: Exact Vasicek mean reversion over time

The script estimates model behavior by simulation; this mini-lab computes the Vasicek conditional moments exactly. For the frozen parameters kappa = 0.65, theta = 0.040, sigma = 0.012, and r0 = 0.035, the expected short rate at horizon t is theta + (r0 - theta) * exp(-kappa * t), with variance sigma^2 * (1 - exp(-2 * kappa * t)) / (2 * kappa). Both are exact for the specified dynamics, with annualized parameters and rates in decimals.

In [ ]:
import numpy as np
import pandas as pd

kappa, theta, sigma, r0 = 0.65, 0.040, 0.012, 0.035  # annual inputs
t = np.array([0.5, 1.0, 2.0, 5.0, 10.0])  # horizons in years
mean = theta + (r0 - theta) * np.exp(-kappa * t)  # exact E[r_t]
var = sigma ** 2 / (2 * kappa) * (1 - np.exp(-2 * kappa * t))  # Var
table = pd.DataFrame({"horizon_y": t, "mean_r": mean.round(5),
                      "std_r": np.sqrt(var).round(5)})
print(table.to_string(index=False))
print("asymptotic mean:", theta, " asymptotic std:",
      round(sigma / np.sqrt(2 * kappa), 5))

### What to observe

The expected rate rises from 0.03639 at the six-month horizon toward the long-run mean of 0.040, reaching 0.03999 by year ten as exp(-kappa * t) dies out. The conditional standard deviation grows from 0.00728 to its asymptotic level of 0.01052. The parameters are teaching inputs rather than a calibrated forecast, so the expectation describes the model, not predicted market rates.

## Exercises

1. **Compare mean-reversion speeds.** Recompute the one-year conditional mean for kappa of 0.25, 0.65, and 1.50 with the other parameters fixed. *Hint:* The expectation is theta + (r0 - theta) * exp(-kappa); a higher kappa pulls the mean faster toward theta.
2. **Check the CIR positivity restriction.** State the parameter condition under which the CIR process stays positive and verify it for the frozen CIR row (kappa 0.70, theta 0.042, sigma 0.080). *Hint:* Positivity requires 2 * kappa * theta to be at least sigma^2.
3. **Explain the minimum-rate column.** Use the script's model summary to explain why the CIR minimum is non-negative while the Vasicek minimum is only positive by chance in this run. *Hint:* The script projects CIR Euler steps to non-negative rates as a numerical safeguard.
4. **Quantify grid sensitivity.** From the grid diagnostics, compute the spread of the six Vasicek estimates and compare it with the reported sampling standard errors. *Hint:* The spread across grids and seeds should be of the same order as the sampling error itself.
5. **Connect to the curve.** Explain why the Hull-White-style process uses an interpolated zero rate as its target and why that is not the same as fitting the initial discount curve. *Hint:* The proxy only sets a drift target; a calibrated Hull-White model chooses a time-dependent drift so that model bond prices match the curve today.

For more detail, see Chapter 12 of *Python & AI for Rates, Bonds, and Credit*.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">